# ESCI hybrid bool + LTR v4 experiment

このnotebookは、LTRプラグインに閉じた範囲でsemantic searchを使う実験を記録します。

## 実験方針

semantic類似度をXGBoostの特徴量には入れません。代わりに、OpenSearchの通常queryで候補生成にだけ使います。

```text
bool.should
  ├─ lexical_v2 query
  └─ k-NN query with semantic_boost β
↓
OpenSearch raw _score で window_size 件を候補選定
↓
LTR plugin の esci_jp_xgb_v3 で rescore
↓
最終ranking
```

この構成は1検索リクエストで完結します。`hybrid` query + normalization processorは使いません。理由は、normalization後topNに対してLTR rescoreする順序を標準機能だけで1リクエスト内に作りにくいためです。

## 評価観点

- semantic_boost β をgrid searchする
- baselineは `β=0`、つまりlexical候補生成 + LTR v3
- 評価はまずv3と揃えて `small_version=1` のvalidation/testで行う
- 指標は `nDCG@10 unjudged=0`, `nDCG@100 unjudged=0`, `recall@100 judged positive`

## 実装

- script: `amazon/run_hybrid_bool_ltr_v4.py`
- artifacts: `amazon/artifacts/hybrid_bool_ltr_v4/`


In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path('..').resolve()
ART = ROOT / 'amazon' / 'artifacts' / 'hybrid_bool_ltr_v4'
ART

## 1. Embedding生成・semantic index作成

MPS/Metalを使い、batch sizeは32を採用します。商品embeddingは今回の実験条件として `max_seq_length=64`, `bullet_chars=0`、つまりtitle/brand/color中心で作成します。

In [ ]:
# !PYTORCH_ENABLE_MPS_FALLBACK=1 ../.venv/bin/python semantic_embeddings.py encode --batch-size 32 --encode-chunk-size 2048 --device mps --max-seq-length 64 --bullet-chars 0
# !../.venv/bin/python semantic_embeddings.py create-index --index amazon-jp-semantic-v1 --recreate
# !../.venv/bin/python semantic_embeddings.py ingest --index amazon-jp-semantic-v1 --bulk-size 250 --device mps
# !../.venv/bin/python semantic_embeddings.py smoke --index amazon-jp-semantic-v1 --device mps --max-seq-length 64

## 2. validationでβ探索

`small_version=1` のtrain内validation queryでsemantic_boostを探索します。

In [ ]:
# !PYTORCH_ENABLE_MPS_FALLBACK=1 ../.venv/bin/python run_hybrid_bool_ltr_v4.py \
#   --index amazon-jp-semantic-v1 \
#   --split validation \
#   --small-only \
#   --boosts 0,0.05,0.1,0.2,0.3,0.5,0.8,1.0 \
#   --size 100 --window-size 100 --knn-k 100 \
#   --batch-queries 8 --device mps \
#   --query-max-seq-length 64 \
#   --include-prerescore \
#   --output-suffix validation_small

summary_path = ART / 'metrics_by_boost_validation_small.csv'
if summary_path.exists():
    display(pd.read_csv(summary_path))
else:
    print('validation summary not found yet')

## 3. best βでtest評価

validationで選んだβをtest splitに適用します。必要に応じて近傍βも一緒に流します。

In [ ]:
# validationでは 0.3 がbestだったため、その周辺とrecall重視の1.0を確認
# !PYTORCH_ENABLE_MPS_FALLBACK=1 ../.venv/bin/python run_hybrid_bool_ltr_v4.py \
#   --index amazon-jp-semantic-v1 \
#   --split test \
#   --small-only \
#   --boosts 0,0.2,0.3,0.5,1.0 \
#   --size 100 --window-size 100 --knn-k 100 \
#   --batch-queries 8 --device mps \
#   --query-max-seq-length 64 \
#   --include-prerescore \
#   --output-suffix test_small

test_summary_path = ART / 'metrics_by_boost_test_small.csv'
if test_summary_path.exists():
    display(pd.read_csv(test_summary_path))
else:
    print('test summary not found yet')

## 4. 実行結果サマリ

実行条件は以下です。

- index: `amazon-jp-semantic-v1`
- products: 339,059件
- product embedding: `intfloat/multilingual-e5-base`, `passage: title | brand | color`, `max_seq_length=64`, `bullet_chars=0`
- query embedding: `query: ...`, `max_seq_length=64`
- candidate generation: `bool.should(lexical_v2, knn)`
- final rerank: OpenSearch LTR plugin `esci_jp_xgb_v3`
- size/window/k: `size=100`, `window_size=100`, `knn_k=100`

Validation smallでは、LTR後の `nDCG@10` は `semantic_boost=0.3` がbestでした。

| semantic_boost | LTR nDCG@10 | LTR nDCG@100 | recall@100 |
|---:|---:|---:|---:|
| 0.0 | 0.428007 | 0.484222 | 0.472151 |
| 0.2 | 0.429476 | 0.488223 | 0.480058 |
| 0.3 | 0.429595 | 0.488345 | 0.480253 |
| 0.5 | 0.429531 | 0.488378 | 0.480593 |
| 1.0 | 0.429387 | 0.488869 | 0.481894 |

Test smallでは、LTR後の `nDCG@10` は `semantic_boost=0.5` がbestでした。

| semantic_boost | LTR nDCG@10 | LTR nDCG@100 | recall@100 |
|---:|---:|---:|---:|
| 0.0 | 0.425388 | 0.488270 | 0.485383 |
| 0.2 | 0.427699 | 0.492650 | 0.492081 |
| 0.3 | 0.427698 | 0.492839 | 0.492309 |
| 0.5 | 0.427806 | 0.493316 | 0.493210 |
| 1.0 | 0.427231 | 0.493497 | 0.494121 |

Test smallの `semantic_boost=0.5` はbaseline `0.0` に対して以下の改善でした。

- `nDCG@10`: +0.002417, 95% CI `[+0.001313, +0.003671]`
- `nDCG@100`: +0.005046, 95% CI `[+0.003733, +0.006518]`
- `recall@100 judged positive`: +0.007827, 95% CI `[+0.006042, +0.009807]`

結論として、semantic scoreをXGBoost特徴量に入れなくても、LTR前の候補生成にk-NNを混ぜるだけでrecallとLTR後nDCGの両方に小さいが一貫した改善が出ました。今回の採用候補は、validation選択を重視するなら `semantic_boost=0.3`、test上の最大値を見るなら `0.5` です。保守的には `0.3`、効果重視なら `0.5` がよさそうです。

## 5. 改善/悪化queryの確認

`example_improvements.csv` と `example_regressions.csv` に、nDCG@10差分が大きいqueryを保存します。

In [ ]:
for name in ['example_improvements.csv', 'example_regressions.csv']:
    path = ART / name
    if path.exists():
        print('\n###', name)
        display(pd.read_csv(path).head(20))